In [ ]:
import re
import hashlib
from dataclasses import dataclass, field
from collections import Counter
from typing import Optional
from pathlib import Path
import sys
sys.path.insert(0, '../src')
from schema import Document, Chunk
from chunker import ABBREVIATIONS, split_sentences, chunk_document
from cadec_loader import DRUG_ALIASES, normalize_drug_name, clean_text, load_cadec

In [ ]:
SAMPLE_POSTS = [
    {
        "drug": "atorvastatin",
        "text": "Been on Lipitor for about 6 months and my legs are killing me. "
                "The muscle pain started around week 3 and keeps getting worse. "
                "My thighs ache constantly, I can barely get up the stairs. "
                "Also having memory problems which I never had before."
    },
    {
        "drug": "atorvastatin",
        "text": "Lipitor worked great for my cholesterol but destroyed my liver. "
                "My enzymes went through the roof at my 3 month check. "
                "Had to stop immediately."
    },
    {
        "drug": "ibuprofen",
        "text": "Took ibuprofen for a week for back pain and ended up in hospital "
                "with a bleeding stomach ulcer. Never had stomach problems before. "
                "Wish someone had warned me about this risk."
    },
    {
        "drug": "ibuprofen",
        "text": "My kidneys were badly affected by long term ibuprofen use. "
                "Taking 600mg three times a day for chronic pain. "
                "Creatinine went up and now I have early stage kidney damage. I am only 42."
    },
    {
        "drug": "duloxetine",
        "text": "Cymbalta was a nightmare to come off. Brain zaps every few minutes, "
                "dizziness, crying constantly. Took 4 months of slow tapering to get off properly. "
                "My doctor had no idea how bad discontinuation syndrome would be."
    },
    {
        "drug": "duloxetine",
        "text": "The nausea on Cymbalta was unbearable for the first month. "
                "Lost 4kg because I could not eat properly. "
                "Depression is better now but that first month was really rough."
    },
    {
        "drug": "diclofenac",
        "text": "Had a heart scare on diclofenac. Chest tightness and shortness of breath "
                "after 3 months of use. Cardiologist said NSAIDs increase cardiac risk "
                "especially in older patients. Stopped immediately."
    },
    {
        "drug": "diclofenac",
        "text": "Diclofenac made my tinnitus so much worse. Already had mild ringing "
                "but after 2 weeks it became loud and constant. "
                "Stopped taking it and slowly improving."
    },
    {
        "drug": "aripiprazole",
        "text": "Abilify gave me an uncontrollable urge to gamble. "
                "Never set foot in a casino and within 2 months losing hundreds weekly. "
                "Doctor never warned me about impulse control issues. "
                "This is a known side effect apparently."
    },
    {
        "drug": "aripiprazole",
        "text": "Severe akathisia on aripiprazole. Cannot sit still, "
                "constant need to move my legs. Feels like internal restlessness I cannot describe. "
                "Worse than the condition it is treating."
    },
]


        

In [ ]:
docs = load_cadec("../data/cadec")
print("Total docs: ", {len(docs)})

In [ ]:
# Case 1: basic splitting
text1 = "Ibuprofen caused nausea. I stopped taking it. My doctor agreed."
print(split_sentences(text1))

# Case 2: abbreviation handling
text2 = "I saw Dr. Smith today. He prescribed 400mg. Take with food."
print(split_sentences(text2))
# expect: 3 sentences, NOT 4 (Dr. should not split)

# Case 3: messy patient text — no terminal punctuation
text3 = "been on lipitor for months and legs are killing me"
print(split_sentences(text3))
# expect: 1 sentence — returned as-is

# Case 4: real CADEC-style post
text4 = ("Cymbalta was a nightmare to come off. "
         "Brain zaps every few minutes, dizziness, crying constantly. "
         "Took 4 months of slow tapering to get off properly.")
print(split_sentences(text4))
# expect: 3 sentences

In [ ]:
all_chunks=[]
for doc in docs:
    chunks= chunk_document(doc, overlap=1, chunk_size = 400)
    all_chunks.extend(chunks)

In [ ]:
print(f"len of docs: {len(docs)}")
print(f"len of all chunks: {len(all_chunks)}")
print(f"Avg chunks/doc: {len(all_chunks)/len(docs):.1f}")


In [ ]:
# Size distribution
lengths = [c.word_count() for c in all_chunks]
print(f"\nChunk word counts:")
print(f"  min  : {min(lengths)}")
print(f"  max  : {max(lengths)}")
print(f"  mean : {sum(lengths)/len(lengths):.1f}")

# By drug
print(f"\nBy drug:")
for drug, count in Counter(c.drug_name for c in all_chunks).most_common():
    print(f"  {drug:<20} {count} chunks")

In [ ]:
print("=== Sample chunks ===\n")
for chunk in all_chunks[:5]:
    print(f"[{chunk.chunk_id}] doc={chunk.doc_id} drug={chunk.drug_name}")
    print(f"sentences: {chunk.metadata.get('sentence_count', '?')}")
    print(f"text: {chunk.text}")
    print()

In [ ]:
all_drugs = Counter(doc.drug_name for doc in  docs)
print("Unique drugs: ", len(all_drugs))
print()
for drug, count in all_drugs.most_common():
    print(f"   {drug:<30}{count} docs")

In [ ]:
tiny_chunks = [c for c in all_chunks if c.word_count()<10]
print("Total tiny chunks: ", len(tiny_chunks))
for c in tiny_chunks[:65]:
    print(f"[{c.word_count()} words] '{c.text}'")

In [ ]:
# Are these tiny chunks standalone documents (1 chunk per doc)
# or are they trailing fragments that should have been merged?

tiny = [c for c in all_chunks if c.word_count() < 10]

standalone = [c for c in tiny if c.chunk_index == 0]
trailing   = [c for c in tiny if c.chunk_index > 0]

print(f"Tiny chunks that are the only chunk in their doc: {len(standalone)}")
print(f"Tiny chunks that are trailing fragments:          {len(trailing)}")

In [ ]:
print("=== CADEC Corpus Summary ===\n")

# Document level
char_lengths = [len(doc.text) for doc in docs]
print(f"Total documents : {len(docs)}")
print(f"Total chunks    : {len(all_chunks)}")
print(f"Avg chunks/doc  : {len(all_chunks)/len(docs):.1f}")

print(f"\nDocument lengths (chars):")
print(f"  min  : {min(char_lengths)}")
print(f"  max  : {max(char_lengths)}")
print(f"  mean : {sum(char_lengths)/len(char_lengths):.0f}")

# Chunk level
chunk_words = [c.word_count() for c in all_chunks]
print(f"\nChunk lengths (words):")
print(f"  min  : {min(chunk_words)}")
print(f"  max  : {max(chunk_words)}")
print(f"  mean : {sum(chunk_words)/len(chunk_words):.1f}")

# Drug distribution
print(f"\nBy drug:")
for drug, count in Counter(c.drug_name for c in all_chunks).most_common():
    print(f"  {drug:<20} {count} chunks")

# Short doc breakdown
short_docs = [d for d in docs if len(d.text) < 100]
print(f"\nDocs under 100 chars : {len(short_docs)} ({len(short_docs)/len(docs)*100:.1f}%)")
print(f"Docs over 400 chars  : {sum(1 for d in docs if len(d.text) > 400)}")